In [2]:
import pandas as pd
import numpy as np

# Load men's and women's regular season detailed results
men_reg = pd.read_csv('data/MRegularSeasonDetailedResults.csv')
women_reg = pd.read_csv('data/WRegularSeasonDetailedResults.csv')
men_tour = pd.read_csv('data/MNCAATourneyDetailedResults.csv')
women_tour = pd.read_csv('data/WNCAATourneyDetailedResults.csv')

# Filter seasons from 2010 onward
men_reg = men_reg[men_reg['Season'] >= 2010]
women_reg = women_reg[women_reg['Season'] >= 2010]
men_tour = men_tour[men_tour['Season'] >= 2010]
women_tour = women_tour[women_tour['Season'] >= 2010]

# Verify the data shapes
print("Men Regular Season shape:", men_reg.shape)
print("Women Regular Season shape:", women_reg.shape)
print("Men Tournament shape:", men_tour.shape)
print("Women Tournament shape:", women_tour.shape)

# Peek at the first few rows of men's regular season data
men_reg.head(3)


Men Regular Season shape: (83674, 34)
Women Regular Season shape: (80626, 34)
Men Tournament shape: (934, 34)
Women Tournament shape: (894, 34)


,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
34074,2010,7,1143,75,1293,70,H,0,24,52,...,21,10,15,11,20,11,17,7,3,24
34075,2010,7,1314,88,1198,72,H,0,34,61,...,23,14,17,13,16,15,20,14,2,18
34076,2010,7,1326,100,1108,60,H,0,39,73,...,17,11,20,11,24,7,13,4,5,16


In [3]:
max_season_men_reg = men_reg['Season'].max()
max_season_women_reg = women_reg['Season'].max()
max_season_men_tour = men_tour['Season'].max()
max_season_women_tour = women_tour['Season'].max()

print("Max season in men's regular season data:", max_season_men_reg)
print("Max season in women's regular season data:", max_season_women_reg)
print("Max season in men's tournament data:", max_season_men_tour)
print("Max season in women's tournament data:", max_season_women_tour)

Max season in men's regular season data: 2025
Max season in women's regular season data: 2025
Max season in men's tournament data: 2024
Max season in women's tournament data: 2024


In [2]:
# Combine men's and women's regular season data
reg_combined = pd.concat([men_reg, women_reg], ignore_index=True)

# Prepare a dataframe of team stats per game for aggregation
# Create a "team perspective" view of each game: one row for each team (winner and loser)
winner_stats = reg_combined.copy()
loser_stats = reg_combined.copy()

# For winner_stats, assign team as WTeam and their stats; similarly for loser_stats
winner_stats['TeamID'] = winner_stats['WTeamID']
winner_stats['OpponentID'] = winner_stats['LTeamID']
winner_stats['TeamScore'] = winner_stats['WScore']
winner_stats['OpponentScore'] = winner_stats['LScore']
winner_stats['TeamLoc'] = winner_stats['WLoc']  # winner's location from WLoc
winner_stats['Win'] = 1  # winner team won the game

loser_stats['TeamID'] = loser_stats['LTeamID']
loser_stats['OpponentID'] = loser_stats['WTeamID']
loser_stats['TeamScore'] = loser_stats['LScore']
loser_stats['OpponentScore'] = loser_stats['WScore']
# Determine location for the losing team: if winner was Home (H), loser was Away, etc.
location_map = {'H': 'A', 'A': 'H', 'N': 'N'}
loser_stats['TeamLoc'] = loser_stats['WLoc'].map(location_map)
loser_stats['Win'] = 0  # loser team lost the game

# Select relevant columns for computing stats
team_cols = ['Season', 'TeamID', 'TeamScore', 'OpponentScore', 'TeamLoc',
             'WFGM','WFGA','WFGM3','WFGA3','WFTM','WFTA','WOR','WDR','WAst','WTO','WStl','WBlk','WPF',
             'LFGM','LFGA','LFGM3','LFGA3','LFTM','LFTA','LOR','LDR','LAst','LTO','LStl','LBlk','LPF', 
             'Win']
# Note: WFGM etc. in loser_stats correspond to opponent's stats (since original losing team stats are in L... columns)
# We'll rename columns appropriately after combining.

winner_stats = winner_stats[team_cols]
loser_stats = loser_stats[team_cols]

# Rename columns in loser_stats to align with winner perspective (so each row has its own team's stats labeled uniformly)
col_rename = {
    'WFGM':'FGM', 'WFGA':'FGA', 'WFGM3':'FGM3', 'WFGA3':'FGA3', 'WFTM':'FTM', 'WFTA':'FTA', 'WOR':'OR', 'WDR':'DR', 'WAst':'Ast', 'WTO':'TO', 'WStl':'Stl', 'WBlk':'Blk', 'WPF':'PF',
    'LFGM':'FGM', 'LFGA':'FGA', 'LFGM3':'FGM3', 'LFGA3':'FGA3', 'LFTM':'FTM', 'LFTA':'FTA', 'LOR':'OR', 'LDR':'DR', 'LAst':'Ast', 'LTO':'TO', 'LStl':'Stl', 'LBlk':'Blk', 'LPF':'PF'
}
# For winner_stats, the W-prefix columns should be considered the team's own stats, and L-prefix are opponent stats.
# For loser_stats, the L-prefix columns are the team's own stats, W-prefix are opponent stats.
# To unify, rename after splitting the data frames.

# Split columns into team and opponent stats for clarity
team_stats_cols = ['FGM','FGA','FGM3','FGA3','FTM','FTA','OR','DR','Ast','TO','Stl','Blk','PF']
opp_stats_cols = ['Opp_FGM','Opp_FGA','Opp_FGM3','Opp_FGA3','Opp_FTM','Opp_FTA','Opp_OR','Opp_DR','Opp_Ast','Opp_TO','Opp_Stl','Opp_Blk','Opp_PF']

# Create separate dataframes for team's own stats and opponent stats for winner and loser frames
winner_team_stats = winner_stats[['Season','TeamID','TeamScore','OpponentScore','TeamLoc','Win']].copy()
loser_team_stats = loser_stats[['Season','TeamID','TeamScore','OpponentScore','TeamLoc','Win']].copy()

# Add team stat columns by mapping from winner_stats and loser_stats
for stat in team_stats_cols:
    winner_team_stats[stat] = winner_stats['W'+stat]  # winner's own stat e.g. WFGM
    loser_team_stats[stat] = loser_stats['L'+stat]    # loser's own stat e.g. LFGM
for stat in team_stats_cols:
    # Opponent's stats: for winner row, opponent stats come from L-prefixed columns; for loser row, from W-prefixed
    opp_stat = 'Opp_' + stat
    winner_team_stats[opp_stat] = winner_stats['L'+stat]
    loser_team_stats[opp_stat] = loser_stats['W'+stat]

# Combine winner and loser team-level stats
team_game_stats = pd.concat([winner_team_stats, loser_team_stats], ignore_index=True)

print("Team-game stats shape:", team_game_stats.shape)
team_game_stats.head(3)


Team-game stats shape: (328600, 32)


,Season,TeamID,TeamScore,OpponentScore,TeamLoc,Win,FGM,FGA,FGM3,FGA3,...,Opp_FGA3,Opp_FTM,Opp_FTA,Opp_OR,Opp_DR,Opp_Ast,Opp_TO,Opp_Stl,Opp_Blk,Opp_PF
0,2010,1143,75,70,H,1,24,52,5,12,...,21,10,15,11,20,11,17,7,3,24
1,2010,1314,88,72,H,1,34,61,4,13,...,23,14,17,13,16,15,20,14,2,18
2,2010,1326,100,60,H,1,39,73,14,33,...,17,11,20,11,24,7,13,4,5,16


In [3]:
# Group by Season and TeamID to aggregate stats
grouped = team_game_stats.groupby(['Season', 'TeamID'])

# Calculate aggregate stats
team_stats = grouped.agg({
    'TeamScore': ['sum','mean'],
    'OpponentScore': ['sum','mean'],
    'Win': 'sum',       # total wins
    'FGM': 'sum', 'FGA': 'sum',
    'FGM3': 'sum','FGA3': 'sum',
    'FTM': 'sum','FTA': 'sum',
    'OR': 'sum','DR': 'sum',
    'TO': 'sum',
    # Opponent stats sums for possessions calculation
    'Opp_OR': 'sum', 'Opp_TO': 'sum', 'Opp_FTA': 'sum'
})
# Flatten MultiIndex columns
team_stats.columns = ['_'.join(col) for col in team_stats.columns]

# Compute derived stats
team_stats['GamesPlayed'] = grouped.size()  # count of games per team
team_stats['Wins'] = team_stats['Win_sum']
team_stats['WinPct'] = team_stats['Wins'] / team_stats['GamesPlayed']

team_stats['AvgPointsFor'] = team_stats['TeamScore_mean']
team_stats['AvgPointsAgainst'] = team_stats['OpponentScore_mean']
team_stats['AvgScoreMargin'] = team_stats['TeamScore_mean'] - team_stats['OpponentScore_mean']

# Field goal and 3-point percentages
team_stats['FGP'] = team_stats['FGM_sum'] / team_stats['FGA_sum']
team_stats['FG3P'] = team_stats['FGM3_sum'] / team_stats['FGA3_sum']
team_stats['FTP'] = team_stats['FTM_sum'] / team_stats['FTA_sum']

# Average rebounds and turnovers per game
team_stats['AvgOR'] = team_stats['OR_sum'] / team_stats['GamesPlayed']
team_stats['AvgDR'] = team_stats['DR_sum'] / team_stats['GamesPlayed']
team_stats['AvgTO'] = team_stats['TO_sum'] / team_stats['GamesPlayed']

# Offensive and Defensive Possessions (approximate)
# Total possessions for team = FGA - OR + TO + 0.44 * FTA (summing these across all games)
team_stats['Possessions'] = team_stats['FGA_sum'] - team_stats['OR_sum'] + team_stats['TO_sum'] + 0.44 * team_stats['FTA_sum']
# For defensive possessions, we can use opponent's stats (should be roughly the same)
team_stats['DefPossessions'] = team_stats['FGA_sum'] - team_stats['OR_sum'] + team_stats['Opp_TO_sum'] + 0.44 * team_stats['Opp_FTA_sum']

# Offensive and Defensive Efficiency (per 100 possessions)
team_stats['OffEff'] = (team_stats['TeamScore_sum'] / team_stats['Possessions']) * 100
team_stats['DefEff'] = (team_stats['OpponentScore_sum'] / team_stats['DefPossessions']) * 100

# Reset index to columns
team_stats = team_stats.reset_index()
print("Team stats shape:", team_stats.shape)
team_stats.head(5)


Team stats shape: (11241, 35)


,Season,TeamID,TeamScore_sum,TeamScore_mean,OpponentScore_sum,OpponentScore_mean,Win_sum,FGM_sum,FGA_sum,FGM3_sum,...,FGP,FG3P,FTP,AvgOR,AvgDR,AvgTO,Possessions,DefPossessions,OffEff,DefEff
0,2010,1102,1613,55.620690,1826,62.965517,8,580,1322,163,...,0.438729,0.310476,0.637363,6.758621,19.862069,12.793103,1697.20,1704.24,95.038888,107.144534
1,2010,1103,2344,71.030303,2193,66.454545,23,828,1902,218,...,0.435331,0.338509,0.665722,13.454545,22.696970,13.666667,2219.64,2258.36,105.602710,97.105864
2,2010,1104,2192,68.500000,2073,64.781250,17,791,1794,175,...,0.440914,0.350000,0.707317,12.062500,23.125000,12.812500,2088.60,2132.52,104.950685,97.208936
3,2010,1105,1468,63.826087,1617,70.304348,8,487,1315,88,...,0.370342,0.282051,0.641390,13.956522,21.608696,15.913043,1638.52,1675.16,89.593047,96.528093
4,2010,1106,1793,64.035714,1860,66.428571,13,594,1498,166,...,0.396529,0.310861,0.646539,13.357143,22.321429,16.214286,1876.76,1877.24,95.536989,99.081630


In [4]:
# Replace infinite and NaN values resulting from divisions (e.g., 0/0) with 0
team_stats = team_stats.replace([np.inf, -np.inf], np.nan).fillna(0)

# Quick sanity check for any remaining NaNs or inf
print("Any NA in team_stats?", team_stats.isna().any().any())
print("Any infinite in team_stats?", np.isinf(team_stats.select_dtypes(include=[float])).any().any())


Any NA in team_stats? False
Any infinite in team_stats? False


In [5]:
# Combine men's and women's tournament results
tourney_combined = pd.concat([men_tour, women_tour], ignore_index=True)
# Separate training and validation by season
train_games = tourney_combined[tourney_combined['Season'] < 2024].copy()  # 2010-2023
val_games = tourney_combined[tourney_combined['Season'] == 2024].copy()   # 2024

print("Training games count:", len(train_games))
print("Validation games count:", len(val_games))

# Create Team1ID, Team2ID and outcome label for each game
def prepare_matchups(df):
    df = df.copy()
    # Determine Team1 and Team2 as the sorted order of the team IDs (to have a consistent reference)
    df['Team1ID'] = df[['WTeamID','LTeamID']].min(axis=1)
    df['Team2ID'] = df[['WTeamID','LTeamID']].max(axis=1)
    # Outcome: 1 if Team1 wins, 0 if Team2 wins
    # (Team1 wins if Team1ID equals WTeamID)
    df['Result'] = (df['Team1ID'] == df['WTeamID']).astype(int)
    # Determine Team1's perspective location feature:
    # If Team1 was home, away, or neutral in that game.
    # We use WLoc and who was winner to deduce Team1's status.
    loc_map = {'H': 1, 'A': -1, 'N': 0}  # encode home as 1, away as -1, neutral as 0
    def get_team1_loc(row):
        # If neutral, return 0 directly
        if row['WLoc'] == 'N':
            return 0
        # If Team1 is winner:
        if row['Team1ID'] == row['WTeamID']:
            # Team1 is winner; WLoc describes Team1's location
            return loc_map.get(row['WLoc'], 0)
        else:
            # Team1 is loser; WLoc describes winner (Team2)'s location
            # If winner was home (H), Team1 was away -> -1; if winner was away (A), Team1 was home -> 1
            return -loc_map.get(row['WLoc'], 0) if row['WLoc'] in ['H','A'] else 0
    df['Team1Loc'] = df.apply(get_team1_loc, axis=1)
    return df[['Season','Team1ID','Team2ID','Team1Loc','Result']]

train_matches = prepare_matchups(train_games)
val_matches = prepare_matchups(val_games)

train_matches.head(3)


Training games count: 1694
Validation games count: 134


,Season,Team1ID,Team2ID,Team1Loc,Result
0,2010,1115,1457,0,1
1,2010,1124,1358,0,1
2,2010,1139,1431,0,1


In [6]:
# Merge team stats for Team1 and Team2 into the matchup data
# First, prepare a merge-friendly team stats (with Season, TeamID as key)
stats_key = team_stats.set_index(['Season','TeamID'])

# Join Team1 stats
train_merged = train_matches.join(stats_key, on=['Season','Team1ID'], rsuffix='_T1')
train_merged = train_merged.join(stats_key, on=['Season','Team2ID'], rsuffix='_T2')
# Drop any rows with missing stats (if a team didn't have stats for some reason)
train_merged = train_merged.dropna()
print("After merging team stats, training shape:", train_merged.shape)

# Define which stats to use for features from team_stats
feature_cols = ['WinPct','AvgPointsFor','AvgPointsAgainst','AvgScoreMargin','FGP','FG3P','FTP',
                'AvgOR','AvgDR','AvgTO','OffEff','DefEff']

# Compute feature differences for Team1 minus Team2
for col in feature_cols:
    train_merged[f'{col}_diff'] = train_merged[f'{col}'] - train_merged[f'{col}_T2']
# (The columns without _T1 or _T2 suffix in train_merged are from Team1 merge due to join, 
# and with _T2 are from Team2 because of the second join)

# Add Team1Loc as a feature (already in train_merged)
features = [f'{col}_diff' for col in feature_cols] + ['Team1Loc']
X_train = train_merged[features]
y_train = train_merged['Result']

print("Training feature sample:")
X_train.head(3)


After merging team stats, training shape: (1694, 71)
Training feature sample:


,WinPct_diff,AvgPointsFor_diff,AvgPointsAgainst_diff,AvgScoreMargin_diff,FGP_diff,FG3P_diff,FTP_diff,AvgOR_diff,AvgDR_diff,AvgTO_diff,OffEff_diff,DefEff_diff,Team1Loc
0,-0.035417,1.839583,2.754167,-0.914583,0.026144,0.039044,0.024015,-0.972917,0.825000,4.737500,0.427698,8.488163,0
1,0.024194,-3.183180,-5.773041,2.589862,0.016923,0.000871,0.024458,0.157834,2.058756,0.571429,0.067707,1.357441,0
2,0.062500,-5.687500,-4.156250,-1.531250,-0.023173,-0.005059,0.075933,-0.656250,-2.843750,-0.750000,1.927139,8.175250,0


In [7]:
feature_cols = [
    'WinPct','AvgPointsFor','AvgPointsAgainst','AvgScoreMargin',
    'FGP','FG3P','FTP','AvgOR','AvgDR','AvgTO','OffEff','DefEff'
]

# STEP 1: Merge team stats for 2024 validation matches
val_merged = val_matches.join(stats_key, on=['Season','Team1ID'], rsuffix='_T1')
val_merged = val_merged.join(stats_key, on=['Season','Team2ID'], rsuffix='_T2')
val_merged = val_merged.dropna()

# STEP 2: Compute diff columns
for col in feature_cols:
    val_merged[f"{col}_diff"] = val_merged[col] - val_merged[f"{col}_T2"]

# Possibly add any additional features you used (e.g., location)
features = [f"{col}_diff" for col in feature_cols] + ["Team1Loc"]

X_val = val_merged[features]
y_val = val_merged["Result"]

print("Validation set shape:", X_val.shape)
print("Features:", X_val.columns.tolist())

Validation set shape: (134, 13)
Features: ['WinPct_diff', 'AvgPointsFor_diff', 'AvgPointsAgainst_diff', 'AvgScoreMargin_diff', 'FGP_diff', 'FG3P_diff', 'FTP_diff', 'AvgOR_diff', 'AvgDR_diff', 'AvgTO_diff', 'OffEff_diff', 'DefEff_diff', 'Team1Loc']


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import log_loss, brier_score_loss, accuracy_score, roc_auc_score, make_scorer

# Define a parameter grid for Random Forest
param_grid = {
    'n_estimators': [100, 200, 500],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 3, 5],
    'max_features': ['sqrt', 'log2', 0.5]  # use fraction or methods for max features
}

# Initialize Random Forest and RandomizedSearchCV
rf_clf = RandomForestClassifier(random_state=42)
scorer = make_scorer(log_loss, greater_is_better=False, needs_proba=True)
random_search = RandomizedSearchCV(rf_clf, param_grid, n_iter=20, scoring=scorer, cv=5, 
                                   random_state=42, n_jobs=-1, verbose=1, refit=True)
random_search.fit(X_train, y_train)

print("Best parameters:", random_search.best_params_)
print("Best CV log loss:", -random_search.best_score_)


Fitting 5 folds for each of 20 candidates, totalling 100 fits


C:\Users\JonMa\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\sklearn\model_selection\_search.py:1108: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan]
  warnings.warn(


Best parameters: {'n_estimators': 100, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': 'log2', 'max_depth': 5}
Best CV log loss: nan


In [9]:
# Retrieve the best model
best_rf = random_search.best_estimator_

# Predict probabilities on the validation set
val_pred_probs = best_rf.predict_proba(X_val)[:, 1]  # probability that Team1 wins

# Compute evaluation metrics
brier = brier_score_loss(y_val, val_pred_probs)
logloss = log_loss(y_val, val_pred_probs)
auc = roc_auc_score(y_val, val_pred_probs)
# For accuracy, we'll use 0.5 threshold to decide winner
val_pred_labels = (val_pred_probs >= 0.5).astype(int)
acc = accuracy_score(y_val, val_pred_labels)

print(f"Validation Brier Score: {brier:.4f}")
print(f"Validation Log Loss: {logloss:.4f}")
print(f"Validation ROC AUC: {auc:.4f}")
print(f"Validation Accuracy: {acc:.4f}")


Validation Brier Score: 0.1996
Validation Log Loss: 0.5776
Validation ROC AUC: 0.7534
Validation Accuracy: 0.6343


In [11]:
# Ensure team_stats has 2025 season regular season stats (if 2025 regular season data was included in our earlier aggregation)
# If not, we would load 2025 regular season detailed results and compute similarly. 
# For this example, assume team_stats already contains Season 2025 stats from whatever partial data is available.

# Load the sample submission which contains all matchup IDs for 2025
submission_example = pd.read_csv('data/SampleSubmissionStage2.csv')
print("Sample submission shape:", submission_example.shape)
submission_example.head(3)


Sample submission shape: (131407, 2)


,ID,Pred
0,2025_1101_1102,0.5
1,2025_1101_1103,0.5
2,2025_1101_1104,0.5


In [12]:
# Parse the IDs into Season, Team1, Team2
submission = submission_example.copy()
submission[['Season','Team1ID','Team2ID']] = submission['ID'].str.split('_', expand=True)
submission['Season'] = submission['Season'].astype(int)
submission['Team1ID'] = submission['Team1ID'].astype(int)
submission['Team2ID'] = submission['Team2ID'].astype(int)

# Merge team stats for Team1 and Team2 for 2025
submission = submission.join(stats_key, on=['Season','Team1ID'], rsuffix='_T1')
submission = submission.join(stats_key, on=['Season','Team2ID'], rsuffix='_T2')

# Compute feature differences
for col in feature_cols:
    submission[f'{col}_diff'] = submission[f'{col}'] - submission[f'{col}_T2']
# Compute Team1Loc for these matchups.
# All 2025 matchups are in the tournament (neutral site), so we set Team1Loc = 0 (since presumably neutral).
submission['Team1Loc'] = 0

# Prepare final feature matrix for prediction
X_test_2025 = submission[features]

# Predict probabilities for Team1 winning each matchup
submission['Pred'] = best_rf.predict_proba(X_test_2025)[:, 1]

# Prepare final submission DataFrame with ID and Pred columns
final_submission = submission[['ID','Pred']]
print("Submission dataframe shape:", final_submission.shape)
final_submission.head(5)


Submission dataframe shape: (131407, 2)


,ID,Pred
0,2025_1101_1102,0.651001
1,2025_1101_1103,0.094079
2,2025_1101_1104,0.114406
3,2025_1101_1105,0.539313
4,2025_1101_1106,0.449339


In [13]:
# Save the submission to CSV
final_submission.to_csv('submission_2025.csv', index=False)


Got it! I will create a structured Jupyter Notebook that:
- **Trains on 2010-2023** and **validates on 2024**.
- **Tunes hyperparameters** for the best-performing model.
- Uses **Brier Score and other metrics** for model evaluation.
- **Combines men’s and women’s datasets** into a single model.
- Outputs predictions in the required **(131407, 2) submission format (ID, Pred)**.
- Incorporates **home/away advantage (WLoc) as a feature**.

I will generate the notebook and let you know once it's ready!

# March Madness Prediction Model

This Jupyter Notebook builds a **March Madness Prediction Model** that combines data from NCAA men's and women's tournaments. We will load and process detailed game results, engineer advanced team statistics, train a Random Forest classifier (with hyperparameter tuning) on past tournaments (2010-2023), validate on the 2024 tournament, and then predict outcomes for the 2025 tournament matchups. The final output will be a submission CSV of 2025 matchups with predicted win probabilities.

## Data Processing

In this section, we load the detailed game results data and engineer features. We use **detailed results CSVs** (which include rich game statistics) instead of compact results to compute advanced team statistics. We will:

- **Load Data**: Read detailed results for both men's and women's games.
- **Compute Team Stats**: Aggregate per-season team statistics (points, shooting percentages, rebounds, etc.) for each team from regular season data.
- **Prepare Training Data**: Build a combined dataset of past tournament matchups (2010-2023) with features for each team and the game outcome.
- **Combine Men’s and Women’s Data**: Merge the men's and women's data into one dataset for training (TeamIDs are unique across men/women, so they can be combined directly).
- **Include Home/Away (WLoc)**: Use the game location as a feature, indicating if a team was home or away, to account for home-court advantage in regular season games.

### Loading Detailed Game Results

Let's load the detailed results for Regular Season and Tournament games for both men's and women's datasets. We'll filter the data to include seasons 2010 onwards (our model training period is 2010-2023, validation 2024). The detailed results include many columns (score, field goals, rebounds, etc.), but we'll select key columns for our analysis.

```python
import pandas as pd
import numpy as np

# Load men's and women's regular season detailed results
men_reg = pd.read_csv('MRegularSeasonDetailedResults.csv')
women_reg = pd.read_csv('WRegularSeasonDetailedResults.csv')
men_tour = pd.read_csv('MNCAATourneyDetailedResults.csv')
women_tour = pd.read_csv('WNCAATourneyDetailedResults.csv')

# Filter seasons from 2010 onward
men_reg = men_reg[men_reg['Season'] >= 2010]
women_reg = women_reg[women_reg['Season'] >= 2010]
men_tour = men_tour[men_tour['Season'] >= 2010]
women_tour = women_tour[women_tour['Season'] >= 2010]

# Verify the data shapes
print("Men Regular Season shape:", men_reg.shape)
print("Women Regular Season shape:", women_reg.shape)
print("Men Tournament shape:", men_tour.shape)
print("Women Tournament shape:", women_tour.shape)

# Peek at the first few rows of men's regular season data
men_reg.head(3)
```

We expect the regular season data to contain columns like: Season, DayNum, WTeamID, WScore, LTeamID, LScore, WLoc, NumOT, WFGM, WFGA, WFGM3, WFGA3, WFTM, WFTA, WOR, WDR, WAst, WTO, WStl, WBlk, WPF, LFGM, LFGA, ... LPF (and similarly for losing team stats). The tournament data has a similar format (likely with WLoc mostly "N" for neutral site games).

### Computing Advanced Team Statistics

Using the regular season detailed results, we will compute advanced statistics for each team per season. This gives us an estimate of each team's performance leading up to the tournament. We will calculate features for **winners and losers correctly** by using the provided W/L stats (rather than comparing scores manually, to avoid any errors). Key steps:

1. **Aggregate Wins and Games**: Count wins and total games for each team to get win percentage (ensuring we count wins from the WTeamID column rather than comparing scores).
2. **Average Points**: Compute average points scored and allowed per game (Offense and Defense).
3. **Shooting Percentages**: Compute field goal percentage (FG%) and three-point percentage (3P%) for each team.
4. **Rebounding and Turnovers**: Compute average offensive rebounds, defensive rebounds, and turnovers.
5. **Offensive/Defensive Efficiency**: Estimate efficiency as points per 100 possessions. We calculate possessions using the formula: `possessions ≈ FGA - OR + TO + 0.44 * FTA` (a standard approximation for basketball possessions).

We'll create a combined regular season dataframe for men and women, then accumulate stats for each team-season. To do this correctly, we'll create two entries per game: one for the winner (with their stats as "Team") and one for the loser (with their stats as "Team"). This way, each team’s performance across all games can be aggregated easily.

```python
# Combine men's and women's regular season data
reg_combined = pd.concat([men_reg, women_reg], ignore_index=True)

# Prepare a dataframe of team stats per game for aggregation
# Create a "team perspective" view of each game: one row for each team (winner and loser)
winner_stats = reg_combined.copy()
loser_stats = reg_combined.copy()

# For winner_stats, assign team as WTeam and their stats; similarly for loser_stats
winner_stats['TeamID'] = winner_stats['WTeamID']
winner_stats['OpponentID'] = winner_stats['LTeamID']
winner_stats['TeamScore'] = winner_stats['WScore']
winner_stats['OpponentScore'] = winner_stats['LScore']
winner_stats['TeamLoc'] = winner_stats['WLoc']  # winner's location from WLoc
winner_stats['Win'] = 1  # winner team won the game

loser_stats['TeamID'] = loser_stats['LTeamID']
loser_stats['OpponentID'] = loser_stats['WTeamID']
loser_stats['TeamScore'] = loser_stats['LScore']
loser_stats['OpponentScore'] = loser_stats['WScore']
# Determine location for the losing team: if winner was Home (H), loser was Away, etc.
location_map = {'H': 'A', 'A': 'H', 'N': 'N'}
loser_stats['TeamLoc'] = loser_stats['WLoc'].map(location_map)
loser_stats['Win'] = 0  # loser team lost the game

# Select relevant columns for computing stats
team_cols = ['Season', 'TeamID', 'TeamScore', 'OpponentScore', 'TeamLoc',
             'WFGM','WFGA','WFGM3','WFGA3','WFTM','WFTA','WOR','WDR','WAst','WTO','WStl','WBlk','WPF',
             'LFGM','LFGA','LFGM3','LFGA3','LFTM','LFTA','LOR','LDR','LAst','LTO','LStl','LBlk','LPF', 
             'Win']
# Note: WFGM etc. in loser_stats correspond to opponent's stats (since original losing team stats are in L... columns)
# We'll rename columns appropriately after combining.

winner_stats = winner_stats[team_cols]
loser_stats = loser_stats[team_cols]

# Rename columns in loser_stats to align with winner perspective (so each row has its own team's stats labeled uniformly)
col_rename = {
    'WFGM':'FGM', 'WFGA':'FGA', 'WFGM3':'FGM3', 'WFGA3':'FGA3', 'WFTM':'FTM', 'WFTA':'FTA', 'WOR':'OR', 'WDR':'DR', 'WAst':'Ast', 'WTO':'TO', 'WStl':'Stl', 'WBlk':'Blk', 'WPF':'PF',
    'LFGM':'FGM', 'LFGA':'FGA', 'LFGM3':'FGM3', 'LFGA3':'FGA3', 'LFTM':'FTM', 'LFTA':'FTA', 'LOR':'OR', 'LDR':'DR', 'LAst':'Ast', 'LTO':'TO', 'LStl':'Stl', 'LBlk':'Blk', 'LPF':'PF'
}
# For winner_stats, the W-prefix columns should be considered the team's own stats, and L-prefix are opponent stats.
# For loser_stats, the L-prefix columns are the team's own stats, W-prefix are opponent stats.
# To unify, rename after splitting the data frames.

# Split columns into team and opponent stats for clarity
team_stats_cols = ['FGM','FGA','FGM3','FGA3','FTM','FTA','OR','DR','Ast','TO','Stl','Blk','PF']
opp_stats_cols = ['Opp_FGM','Opp_FGA','Opp_FGM3','Opp_FGA3','Opp_FTM','Opp_FTA','Opp_OR','Opp_DR','Opp_Ast','Opp_TO','Opp_Stl','Opp_Blk','Opp_PF']

# Create separate dataframes for team's own stats and opponent stats for winner and loser frames
winner_team_stats = winner_stats[['Season','TeamID','TeamScore','OpponentScore','TeamLoc','Win']].copy()
loser_team_stats = loser_stats[['Season','TeamID','TeamScore','OpponentScore','TeamLoc','Win']].copy()

# Add team stat columns by mapping from winner_stats and loser_stats
for stat in team_stats_cols:
    winner_team_stats[stat] = winner_stats['W'+stat]  # winner's own stat e.g. WFGM
    loser_team_stats[stat] = loser_stats['L'+stat]    # loser's own stat e.g. LFGM
for stat in team_stats_cols:
    # Opponent's stats: for winner row, opponent stats come from L-prefixed columns; for loser row, from W-prefixed
    opp_stat = 'Opp_' + stat
    winner_team_stats[opp_stat] = winner_stats['L'+stat]
    loser_team_stats[opp_stat] = loser_stats['W'+stat]

# Combine winner and loser team-level stats
team_game_stats = pd.concat([winner_team_stats, loser_team_stats], ignore_index=True)

print("Team-game stats shape:", team_game_stats.shape)
team_game_stats.head(3)
```

In the code above, we:
- Created `winner_team_stats` and `loser_team_stats` DataFrames representing each game from the perspective of each team.
- Each row in `team_game_stats` now has the team's own stats (`FGM`, `FGA`, etc.), opponent's stats (`Opp_FGM`, `Opp_FGA`, etc.), the points scored (`TeamScore`) and allowed (`OpponentScore`), whether the team won (`Win`), and the location from the team's perspective (`TeamLoc`: Home **H**, Away **A**, or Neutral **N**).

Now, we'll aggregate these per-team-per-season. For each team and season, we compute:
- **Games Played** (`GP`): number of games.
- **Wins**: number of wins.
- **Win Percentage**: wins / games.
- **Average Points For/Against**: mean of TeamScore and OpponentScore.
- **Average Scoring Margin**: (points for - points against) per game.
- **Field Goal % (FGP)**: total FGM / total FGA.
- **3-Point % (FG3P)**: total FGM3 / total FGA3.
- **Free Throw % (FTP)**: total FTM / total FTA.
- **Average Rebounds (Off/Def)**: mean OR and mean DR.
- **Average Turnovers**: mean TO.
- **Offensive Efficiency (OffEff)**: (total points for / total possessions) * 100.
- **Defensive Efficiency (DefEff)**: (total points against / total opponent possessions) * 100.

We will calculate possessions for each team in each game using the formula:
\[ \text{Possessions} \approx \text{FGA} - \text{OR} + \text{TO} + 0.44 \times \text{FTA} \]
We'll sum these across games to get total possessions.

```python
# Group by Season and TeamID to aggregate stats
grouped = team_game_stats.groupby(['Season', 'TeamID'])

# Calculate aggregate stats
team_stats = grouped.agg({
    'TeamScore': ['sum','mean'],
    'OpponentScore': ['sum','mean'],
    'Win': 'sum',       # total wins
    'FGM': 'sum', 'FGA': 'sum',
    'FGM3': 'sum','FGA3': 'sum',
    'FTM': 'sum','FTA': 'sum',
    'OR': 'sum','DR': 'sum',
    'TO': 'sum',
    # Opponent stats sums for possessions calculation
    'Opp_OR': 'sum', 'Opp_TO': 'sum', 'Opp_FTA': 'sum'
})
# Flatten MultiIndex columns
team_stats.columns = ['_'.join(col) for col in team_stats.columns]

# Compute derived stats
team_stats['GamesPlayed'] = grouped.size()  # count of games per team
team_stats['Wins'] = team_stats['Win_sum']
team_stats['WinPct'] = team_stats['Wins'] / team_stats['GamesPlayed']

team_stats['AvgPointsFor'] = team_stats['TeamScore_mean']
team_stats['AvgPointsAgainst'] = team_stats['OpponentScore_mean']
team_stats['AvgScoreMargin'] = team_stats['TeamScore_mean'] - team_stats['OpponentScore_mean']

# Field goal and 3-point percentages
team_stats['FGP'] = team_stats['FGM_sum'] / team_stats['FGA_sum']
team_stats['FG3P'] = team_stats['FGM3_sum'] / team_stats['FGA3_sum']
team_stats['FTP'] = team_stats['FTM_sum'] / team_stats['FTA_sum']

# Average rebounds and turnovers per game
team_stats['AvgOR'] = team_stats['OR_sum'] / team_stats['GamesPlayed']
team_stats['AvgDR'] = team_stats['DR_sum'] / team_stats['GamesPlayed']
team_stats['AvgTO'] = team_stats['TO_sum'] / team_stats['GamesPlayed']

# Offensive and Defensive Possessions (approximate)
# Total possessions for team = FGA - OR + TO + 0.44 * FTA (summing these across all games)
team_stats['Possessions'] = team_stats['FGA_sum'] - team_stats['OR_sum'] + team_stats['TO_sum'] + 0.44 * team_stats['FTA_sum']
# For defensive possessions, we can use opponent's stats (should be roughly the same)
team_stats['DefPossessions'] = team_stats['FGA_sum'] - team_stats['OR_sum'] + team_stats['Opp_TO_sum'] + 0.44 * team_stats['Opp_FTA_sum']

# Offensive and Defensive Efficiency (per 100 possessions)
team_stats['OffEff'] = (team_stats['TeamScore_sum'] / team_stats['Possessions']) * 100
team_stats['DefEff'] = (team_stats['OpponentScore_sum'] / team_stats['DefPossessions']) * 100

# Reset index to columns
team_stats = team_stats.reset_index()
print("Team stats shape:", team_stats.shape)
team_stats.head(5)
```

We have now created a `team_stats` DataFrame where each row corresponds to a unique (Season, TeamID) and contains the aggregated stats for that team in that season. We included metrics like win percentage, average points, shooting percentages, rebound and turnover averages, and efficiency ratings.

**Handling Missing/Infinite Values**: In some cases, a stat might be undefined (e.g., a team attempted zero 3-pointers, making 3P% division by zero). We address this by replacing infinite or NaN values with a neutral value (0 or appropriate fill). For example, if a team never attempted a 3-point shot, we'll set its 3P% to 0.0.

```python
# Replace infinite and NaN values resulting from divisions (e.g., 0/0) with 0
team_stats = team_stats.replace([np.inf, -np.inf], np.nan).fillna(0)

# Quick sanity check for any remaining NaNs or inf
print("Any NA in team_stats?", team_stats.isna().any().any())
print("Any infinite in team_stats?", np.isinf(team_stats.select_dtypes(include=[float])).any().any())
```

At this point, `team_stats` contains the advanced features for each team and season, and we've ensured there are no missing or infinite values that could cause issues.

### Preparing the Training Dataset (2010-2023) and Validation Set (2024)

Now we will prepare the dataset for model training and validation. Each data point will represent a matchup between two teams in a tournament game, with the features derived from the teams' season stats and the label indicating which team won.

**Steps to prepare training/validation data**:
1. **Combine Tournament Data**: Concatenate men's and women's tournament game results.
2. **Select Training vs Validation**: We will use games from 2010-2023 as training data and games from 2024 as the validation set.
3. **Create Matchup Rows**: For each game, define Team1 and Team2 in a consistent order (e.g., Team1ID < Team2ID to avoid duplicate ordering). Determine the outcome label (1 if Team1 won, 0 if Team2 won).
4. **Merge Features**: Attach the precomputed team statistics for Team1 and Team2 for that season. Compute feature differences (Team1 stat minus Team2 stat) and include home/away indicator for Team1.
5. **Ensure Correct Aggregation**: We use the explicit winners in the data (not score comparison) to mark outcomes, so the label is accurate.

Let's build the training and validation DataFrames:

```python
# Combine men's and women's tournament results
tourney_combined = pd.concat([men_tour, women_tour], ignore_index=True)
# Separate training and validation by season
train_games = tourney_combined[tourney_combined['Season'] < 2024].copy()  # 2010-2023
val_games = tourney_combined[tourney_combined['Season'] == 2024].copy()   # 2024

print("Training games count:", len(train_games))
print("Validation games count:", len(val_games))

# Create Team1ID, Team2ID and outcome label for each game
def prepare_matchups(df):
    df = df.copy()
    # Determine Team1 and Team2 as the sorted order of the team IDs (to have a consistent reference)
    df['Team1ID'] = df[['WTeamID','LTeamID']].min(axis=1)
    df['Team2ID'] = df[['WTeamID','LTeamID']].max(axis=1)
    # Outcome: 1 if Team1 wins, 0 if Team2 wins
    # (Team1 wins if Team1ID equals WTeamID)
    df['Result'] = (df['Team1ID'] == df['WTeamID']).astype(int)
    # Determine Team1's perspective location feature:
    # If Team1 was home, away, or neutral in that game.
    # We use WLoc and who was winner to deduce Team1's status.
    loc_map = {'H': 1, 'A': -1, 'N': 0}  # encode home as 1, away as -1, neutral as 0
    def get_team1_loc(row):
        # If neutral, return 0 directly
        if row['WLoc'] == 'N':
            return 0
        # If Team1 is winner:
        if row['Team1ID'] == row['WTeamID']:
            # Team1 is winner; WLoc describes Team1's location
            return loc_map.get(row['WLoc'], 0)
        else:
            # Team1 is loser; WLoc describes winner (Team2)'s location
            # If winner was home (H), Team1 was away -> -1; if winner was away (A), Team1 was home -> 1
            return -loc_map.get(row['WLoc'], 0) if row['WLoc'] in ['H','A'] else 0
    df['Team1Loc'] = df.apply(get_team1_loc, axis=1)
    return df[['Season','Team1ID','Team2ID','Team1Loc','Result']]

train_matches = prepare_matchups(train_games)
val_matches = prepare_matchups(val_games)

train_matches.head(3)
```

In the code above, we:
- Determined `Team1ID` and `Team2ID` as the smaller and larger team IDs for each matchup (this consistent ordering is crucial for aligning with the submission format and preventing duplicate entries).
- Created the `Result` label: `1` if Team1 won, `0` if Team2 won.
- Encoded the home/away feature from `WLoc` as `Team1Loc`: values {1, -1, 0} indicating Team1 was home (+1), away (-1), or neutral (0). We derived this by considering who was home (WLoc) and whether Team1 was the winner or loser. This incorporates the **home-court advantage** info for each game in training.
  
Now, we will merge the team statistics for Team1 and Team2 into these matchup records. We will then create features, mostly as differences between Team1 and Team2 stats (e.g., `WinPct_diff = Team1.WinPct - Team2.WinPct`). Using differences helps the model focus on the relative strength between teams. We will also include some raw values if needed (the model could potentially learn non-linear interactions, but differences should suffice in most cases).

```python
# Merge team stats for Team1 and Team2 into the matchup data
# First, prepare a merge-friendly team stats (with Season, TeamID as key)
stats_key = team_stats.set_index(['Season','TeamID'])

# Join Team1 stats
train_merged = train_matches.join(stats_key, on=['Season','Team1ID'], rsuffix='_T1')
train_merged = train_merged.join(stats_key, on=['Season','Team2ID'], rsuffix='_T2')
# Drop any rows with missing stats (if a team didn't have stats for some reason)
train_merged = train_merged.dropna()
print("After merging team stats, training shape:", train_merged.shape)

# Define which stats to use for features from team_stats
feature_cols = ['WinPct','AvgPointsFor','AvgPointsAgainst','AvgScoreMargin','FGP','FG3P','FTP',
                'AvgOR','AvgDR','AvgTO','OffEff','DefEff']

# Compute feature differences for Team1 minus Team2
for col in feature_cols:
    train_merged[f'{col}_diff'] = train_merged[f'{col}'] - train_merged[f'{col}_T2']
# (The columns without _T1 or _T2 suffix in train_merged are from Team1 merge due to join, 
# and with _T2 are from Team2 because of the second join)

# Add Team1Loc as a feature (already in train_merged)
features = [f'{col}_diff' for col in feature_cols] + ['Team1Loc']
X_train = train_merged[features]
y_train = train_merged['Result']

print("Training feature sample:")
X_train.head(3)
```

We repeated the merging process for the validation set (2024 games):

```python
# Merge team stats for validation set (2024)
val_merged = val_matches.join(stats_key, on=['Season','Team1ID'], rsuffix='_T1')
val_merged = val_merged.join(stats_key, on=['Season','Team2ID'], rsuffix='_T2')
val_merged = val_merged.dropna()
X_val = val_merged[features]
y_val = val_merged['Result']
print("Validation set size:", X_val.shape)
```

By this point, we have:
- `X_train`: Feature matrix for 2010-2023 tournament games (men & women combined).
- `y_train`: Outcome (Team1 win or not) for those games.
- `X_val`: Feature matrix for 2024 tournament games.
- `y_val`: Actual outcomes for 2024 games (to validate our model).

Crucially, **features are aligned between training and test sets** because we constructed them using the same process and set of stats. By using the `features` list consistently for train, validation, and later for test (2025), we ensure the model sees the same feature columns. We also handled missing values earlier, so there should be no NaNs or infinite values in `X_train` or `X_val`.

## Model Training & Validation

With the processed dataset ready, we train a machine learning model to predict the outcome of a matchup. We choose a **RandomForestClassifier** for its robustness and ability to handle feature interactions. We will perform **hyperparameter tuning** using cross-validation on the training set to optimize the model's performance.

We will evaluate the model using the **Brier Score** (which measures the mean squared error of the predicted probabilities) and other metrics like Log Loss and accuracy. The Brier Score is a proper metric for probabilistic forecasts – lower Brier Score indicates better calibrated predictions. We also track accuracy (how often the predicted winner is correct) just for reference, and Log Loss which is commonly used in competitions for evaluating probability predictions.

### Hyperparameter Tuning for Random Forest

Let's define a hyperparameter search space and use `RandomizedSearchCV` (or GridSearchCV with a limited grid) to find a good set of parameters for the Random Forest. We'll tune parameters such as:
- `n_estimators`: number of trees in the forest.
- `max_depth`: maximum depth of each tree.
- `min_samples_split` and `min_samples_leaf`: to control tree complexity.
- `max_features`: number of features to consider when looking for the best split.

We will use 5-fold cross-validation on the training set (2010-2023 games) for the search, optimizing for a scoring metric. We'll use **neg_log_loss** or **neg_brier_score_loss** as the scoring to encourage good probability predictions. Here, let's use log loss for tuning (it correlates with Brier Score minimization as well).

```python
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import log_loss, brier_score_loss, accuracy_score, roc_auc_score, make_scorer

# Define a parameter grid for Random Forest
param_grid = {
    'n_estimators': [100, 200, 500],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 3, 5],
    'max_features': ['sqrt', 'log2', 0.5]  # use fraction or methods for max features
}

# Initialize Random Forest and RandomizedSearchCV
rf_clf = RandomForestClassifier(random_state=42)
scorer = make_scorer(log_loss, greater_is_better=False, needs_proba=True)
random_search = RandomizedSearchCV(rf_clf, param_grid, n_iter=20, scoring=scorer, cv=5, 
                                   random_state=42, n_jobs=-1, verbose=1, refit=True)
random_search.fit(X_train, y_train)

print("Best parameters:", random_search.best_params_)
print("Best CV log loss:", -random_search.best_score_)
```

The RandomizedSearch will try 20 random combinations of the parameter grid and output the best set found. We set `refit=True`, so `random_search.best_estimator_` will be the model refit on the entire training set with the best parameters.

### Model Evaluation on Validation Set (2024)

Now, let's evaluate how our tuned model performs on the 2024 tournament games (which the model has never seen). We'll compute the **Brier Score**, **Log Loss**, **Accuracy**, and **ROC AUC** for the validation set predictions.

```python
# Retrieve the best model
best_rf = random_search.best_estimator_

# Predict probabilities on the validation set
val_pred_probs = best_rf.predict_proba(X_val)[:, 1]  # probability that Team1 wins

# Compute evaluation metrics
brier = brier_score_loss(y_val, val_pred_probs)
logloss = log_loss(y_val, val_pred_probs)
auc = roc_auc_score(y_val, val_pred_probs)
# For accuracy, we'll use 0.5 threshold to decide winner
val_pred_labels = (val_pred_probs >= 0.5).astype(int)
acc = accuracy_score(y_val, val_pred_labels)

print(f"Validation Brier Score: {brier:.4f}")
print(f"Validation Log Loss: {logloss:.4f}")
print(f"Validation ROC AUC: {auc:.4f}")
print(f"Validation Accuracy: {acc:.4f}")
```

The **Brier Score** is the mean squared error of the probability predictions. For example, a Brier score of 0.2000 indicates some calibration error (0 would be perfect). Log Loss penalizes confident wrong predictions, so lower is better. ROC AUC gives a threshold-independent measure of ranking predictions (1.0 is perfect, 0.5 is random). Accuracy here simply tells how often the model's 50% cutoff guess was correct on winners.

We expect a well-tuned model might have a Brier score in a reasonable range (e.g., around 0.2 or lower is good for this kind of task) and an AUC significantly above 0.5 (since 0.5 is random guessing). Accuracy might be around e.g. 70% if the model picks favorites correctly often (just as a reference, not the primary goal since we care about probabilities).

**Note:** If there's room for improvement, consider including additional features like team seed differences or other advanced metrics (e.g., KenPom ratings if available). In this notebook, we've restricted to stats derived from game data for simplicity. Despite combining men's and women's data, the model should handle them together since TeamID and stats are in one space. 

We have also made sure to not leak any future data: all team stats for 2024 and 2025 are computed from regular season games of those years (no tournament outcomes involved in features). This ensures our validation is genuine and our eventual 2025 predictions are based only on data available before the 2025 tournament.

## Predicting 2025 Matchups and Creating Submission

Finally, we use the trained model to predict the outcomes of the 2025 tournament matchups. The submission requires a probability for each possible matchup in the format **"Season_Team1ID_Team2ID"** with Team1ID < Team2ID, and a probability that Team1 wins.

**Steps**:
1. **Prepare 2025 Team Stats**: Compute stats for the 2025 season from regular season data (similar to what we did for previous seasons). If our `team_stats` DataFrame already includes 2025 (because we likely included up to 2025 in the regular season data), we can use that.
2. **Load Sample Submission**: The sample submission file (often provided) contains all possible matchup IDs for 2025 (for both men's and women's brackets combined). We'll load this to get the list of matchups to predict.
3. **Generate Features for 2025 Matchups**: For each matchup (Team1, Team2), retrieve the team stats from 2025 and compute the feature differences just like in training.
4. **Predict Probabilities**: Use the trained model to predict the probability that Team1 wins.
5. **Output CSV**: Create a DataFrame with `ID` and `Pred` columns and save to CSV. The expected shape is (131407, 2) – meaning there are 131,407 possible matchups listed, and we provide a probability for each.

Let's proceed with these steps:

```python
# Ensure team_stats has 2025 season regular season stats (if 2025 regular season data was included in our earlier aggregation)
# If not, we would load 2025 regular season detailed results and compute similarly. 
# For this example, assume team_stats already contains Season 2025 stats from whatever partial data is available.

# Load the sample submission which contains all matchup IDs for 2025
submission_example = pd.read_csv('SampleSubmissionStage2.csv')
print("Sample submission shape:", submission_example.shape)
submission_example.head(3)
```

The sample submission file has an `ID` column with entries like "2025_1101_1102" and a placeholder `Pred` value. We need to compute `Pred`. Let's parse the IDs and compute features for each matchup:

```python
# Parse the IDs into Season, Team1, Team2
submission = submission_example.copy()
submission[['Season','Team1ID','Team2ID']] = submission['ID'].str.split('_', expand=True)
submission['Season'] = submission['Season'].astype(int)
submission['Team1ID'] = submission['Team1ID'].astype(int)
submission['Team2ID'] = submission['Team2ID'].astype(int)

# Merge team stats for Team1 and Team2 for 2025
submission = submission.join(stats_key, on=['Season','Team1ID'], rsuffix='_T1')
submission = submission.join(stats_key, on=['Season','Team2ID'], rsuffix='_T2')

# Compute feature differences
for col in feature_cols:
    submission[f'{col}_diff'] = submission[f'{col}'] - submission[f'{col}_T2']
# Compute Team1Loc for these matchups.
# All 2025 matchups are in the tournament (neutral site), so we set Team1Loc = 0 (since presumably neutral).
submission['Team1Loc'] = 0

# Prepare final feature matrix for prediction
X_test_2025 = submission[features]

# Predict probabilities for Team1 winning each matchup
submission['Pred'] = best_rf.predict_proba(X_test_2025)[:, 1]

# Prepare final submission DataFrame with ID and Pred columns
final_submission = submission[['ID','Pred']]
print("Submission dataframe shape:", final_submission.shape)
final_submission.head(5)
```

We set `Team1Loc = 0` for all matchups in 2025 because March Madness games are generally played at neutral venues. (If any matchup had a theoretical home advantage, it would be reflected in WLoc, but in the tournament this is typically not applicable.)

The final `final_submission` DataFrame should have the shape (131407, 2) as required, containing every possible matchup ID and our predicted probability. We can now save this to a CSV file for submission:

```python
# Save the submission to CSV
final_submission.to_csv('submission_2025.csv', index=False)
```

This CSV will have two columns: **ID** (like "2025_1101_1102") and **Pred** (the probability that Team1 wins). 

## Conclusion

We have successfully built a March Madness prediction model that:
- Uses detailed game data to compute advanced team features (offensive/defensive stats, efficiencies, etc.) for each team.
- Combines men's and women's tournament data into a single training set to enrich model training.
- Incorporates home/away location as a feature to adjust for home-court advantage in games.
- Trains a Random Forest classifier on 14 years of historical data with hyperparameter tuning to optimize performance.
- Validates the model on the 2024 tournament, evaluating it with Brier Score, Log Loss, and other metrics.
- Generates predictions for all potential 2025 tournament matchups in the required submission format.

Throughout this process, we ensured that features are consistently generated for training, validation, and test sets, and handled any missing or infinite values. The model's predictions can be further improved by incorporating additional data (such as team seed rankings or elo ratings), but this notebook provides a solid baseline approach for March Madness outcome prediction. Good luck with the 2025 bracket predictions!